# Few-Shot Prompting (예시 기반 프롬프트) 실습

오늘 실습한 내용은 **Few-Shot Prompting**이다. 질문에 답하는 방법(추론 과정)을 몇 가지 예시로 미리 보여준 뒤, 실제 질문을 던지면 LLM이 그 예시들의 패턴을 따라서 답변하도록 유도하는 기법이다.

여기서 사용한 예시들은 복잡한 질문을 여러 개의 하위 질문(추가 질문)으로 쪼개고, 각 하위 질문에 대한 중간 답변을 거쳐 최종 답변에 도달하는 **Self-Ask(자문자답) 스타일**로 구성되어 있다.

## 1. 환경변수 로드

`.env` 파일에 저장해 둔 `OPENAI_API_KEY` 등의 환경변수를 불러온다. `load_dotenv()`가 성공하면 `True`를 반환한다.

In [2]:
from dotenv import load_dotenv

# .env 파일에 저장된 환경변수(OPENAI_API_KEY 등)를 현재 프로세스 환경변수로 불러온다.
# 성공하면 True를 반환한다.
load_dotenv()

True

## 2. LLM 준비 및 스트리밍 응답 기본 동작 확인

`ChatOpenAI` 모델을 하나 생성해 두고, Few-Shot 예시 없이 일반 질문을 그대로 스트리밍으로 물어봐서 기본 동작을 먼저 확인한다.

- `temperature=0` : 답변의 무작위성을 최소화해서 매번 비슷하고 일관된 답변이 나오도록 한다.
- `model="gpt-4.1"` : 사용할 모델 지정.
- `llm.stream(question)` : 질문을 스트리밍 방식으로 호출해서, 답변을 토큰(chunk) 단위로 순서대로 받는다.

In [15]:
from langchain_openai import ChatOpenAI

# temperature=0 : 출력의 무작위성을 낮춰 일관된 답변을 얻는다.
# model="gpt-4.1" : 사용할 OpenAI 모델 지정.
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4.1",
)

question = "대한민국의 수도는 뭐야?"

# .stream()은 답변을 한 번에 받는 대신, 생성되는 대로 chunk(토큰) 단위로 흘려보내준다.
# 각 chunk.content를 이어 출력하면 답변이 실시간으로 타이핑되듯 출력된다.
answer = llm.stream(question)
for chunk in answer:
    print(chunk.content, end="", flush=True)

대한민국의 수도는 서울입니다.

## 3. Few-Shot 예시(examples) 데이터 준비

`FewShotPromptTemplate`에 넣어줄 예시 목록을 만든다. 각 예시는 `question`(질문)과 `answer`(답변) 한 쌍으로 구성되며, `answer`는 아래와 같은 **Self-Ask 형식**을 따른다.

```
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: (하위 질문 1)
중간 답변: (하위 질문 1에 대한 답)
추가 질문: (하위 질문 2)
중간 답변: (하위 질문 2에 대한 답)
...
최종 답변은: (최종 결론)
```

이렇게 "질문을 잘게 쪼개고 단계적으로 답하는" 예시를 여러 개 보여주면, 이후 새로운 질문에도 LLM이 같은 방식으로 단계적 추론을 하도록 유도할 수 있다 (Chain-of-Thought와 유사한 효과).

In [6]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Few-Shot 예시 데이터.
# 각 answer는 "추가 질문 -> 중간 답변"을 반복해서 최종 답변에 도달하는
# Self-Ask(자문자답) 방식으로 작성되어 있다.
examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
""",
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
""",
    },
]

## 4. 예시 1개를 실제 프롬프트 문자열로 포맷해보기

`FewShotPromptTemplate`에 각 예시를 어떤 텍스트 형태로 끼워 넣을지 정하는 것이 `example_prompt`이다. `{question}`, `{answer}` 자리에 각 예시의 값이 채워진다.

먼저 예시 하나(`examples[0]`)만 가지고 실제로 어떤 문자열이 만들어지는지 확인해본다.

In [8]:
# 예시 하나를 "Question:\n...\nAnswer:\n..." 형태의 텍스트로 바꿔주는 템플릿.
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

# examples[0] 딕셔너리를 그대로 언패킹(**)해서 {question}, {answer}에 채워 넣는다.
print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인



## 5. FewShotPromptTemplate으로 예시 + 실제 질문 합치기

`FewShotPromptTemplate`은 `examples`에 있는 예시들을 `example_prompt` 형식으로 하나씩 나열한 뒤, 마지막에 `suffix`로 지정한 실제 질문을 이어붙여 최종 프롬프트를 만든다.

- `examples` : 앞서 만든 Self-Ask 예시 4개
- `example_prompt` : 각 예시를 렌더링할 템플릿
- `suffix="Question:\n{question}\nAnswer:"` : 예시들 뒤에 붙는, 실제로 답을 구해야 할 새 질문. **의도적으로 답(Answer) 없이 질문까지만** 작성해서, LLM이 그 뒤를 이어서(=예시들과 같은 패턴으로) 답하도록 유도한다.
- `input_variables=["question"]` : `format()` 호출 시 채워야 하는 변수 목록.

`final_prompt`를 출력해보면 예시 4개 + 새로운 질문이 하나의 긴 프롬프트로 합쳐진 것을 확인할 수 있다.

In [12]:
prompt = FewShotPromptTemplate(
    examples=examples,             # 위에서 만든 Self-Ask 예시들
    example_prompt=example_prompt, # 예시 1개를 텍스트로 렌더링하는 템플릿
    suffix="Question:\n{question}\nAnswer:",  # 예시들 뒤에 붙을 새 질문 (답은 비워둠)
    input_variables=["question"],
)

question = "Google이 창린된 연도에 Bill Gates의 나이는 몇 살인가요?"
# 예시 4개 + 새 질문이 합쳐진 최종 프롬프트 문자열을 만든다.
final_prompt = prompt.format(question=question)
print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인


Question:
네이버의 창립자는 언제 태어났나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일


Question:
율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군


Question:
올드보이와 기생충의 감독이 같은 나라 출신인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예


Question:
Google이 창린된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


## 6. 완성된 프롬프트를 LLM에 그대로 스트리밍 호출

방금 만든 `final_prompt`(예시 4개 + 새 질문) 를 `llm.stream()`에 그대로 넘긴다. LLM은 앞선 예시들의 "추가 질문 → 중간 답변 → 최종 답변" 패턴을 따라, 새 질문에 대해서도 단계적으로 추론한 뒤 답을 낸다.

실제로 출력 결과를 보면 "Google 창립 연도(1998) 확인 → Bill Gates 생년(1955) 확인 → 나이 계산(43세)"처럼, 하위 질문으로 쪼개서 단계적으로 답하는 것을 볼 수 있다.

In [20]:
# 예시가 포함된 final_prompt를 그대로 LLM에 넘겨 스트리밍으로 답을 받는다.
# LLM은 예시들의 "추가 질문 -> 중간 답변 -> 최종 답변" 패턴을 그대로 이어서 답변한다.
answer = llm.stream(final_prompt)
for chunk in answer:
    print(chunk.content, end="", flush=True)

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년에 창립되었습니다.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: 1998년에 Bill Gates의 나이는 몇 살이었나요?  
중간 답변: 1998년 - 1955년 = 43년. Bill Gates는 1998년에 43세였습니다.  
최종 답변은: 43세

## 7. 체인(prompt | llm | StrOutputParser)으로 재구성

앞에서는 `prompt.format()`으로 직접 문자열을 만들고 `llm.stream(final_prompt)`을 호출했다. 이번에는 LangChain의 파이프(`|`) 문법으로 `prompt | llm | StrOutputParser()`를 연결해서 체인으로 만든다.

- `suffix="Question:\n{question}\nAnswer"` : 이번에는 `Answer` 뒤에 콜론(`:`)을 빼고 실험.
- `chain.stream({"question": ...})` : `format()`을 직접 호출하지 않고, 체인에 변수 딕셔너리를 넘기면 `FewShotPromptTemplate` → `llm` → `StrOutputParser` 순서로 자동으로 처리된다.
- `StrOutputParser()` : LLM이 반환하는 메시지 청크(`AIMessageChunk`)에서 텍스트(`content`)만 뽑아 순수 문자열 스트림으로 바꿔준다. 그래서 이전 셀과 달리 `chunk.content`가 아니라 `chunk` 자체를 바로 출력하면 된다.

In [21]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer",  # 콜론(:) 없이 구성해봄
    input_variables=["question"],
)

# prompt | llm | StrOutputParser() : Few-Shot 프롬프트 -> LLM 호출 -> 문자열 파싱까지
# 하나의 체인으로 연결. StrOutputParser 덕분에 결과가 바로 순수 문자열로 나온다.
chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)

# StrOutputParser를 거쳤기 때문에 chunk.content가 아니라 chunk 자체가 문자열이다.
for chunk in answer:
    print(chunk, end="", flush=True)

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년에 창립되었습니다.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: 1998년에 Bill Gates의 나이는 몇 살이었나요?  
중간 답변: 1998년에서 1955년을 빼면 43이므로, Bill Gates는 1998년에 43세였습니다.  
최종 답변은: 43세